# 00b - Milton local clustering (rule B1, 100-mi)

Merge **small** Milton counties (NCHS 4-6: small-metro / micropolitan / noncore) with same-NCHS contiguous
neighbours; keep metros (NCHS 1-3) standalone. 34 counties -> **30 clusters**. NOTE: clustered Milton is the
**appendix robustness** set only — the PRIMARY Milton analysis is county-level (N=34).

Outputs:
- `results/local_level/milton_clustered_100mi/county_cluster_assignments.csv` (GEOID->cluster)
- `results/milton_clustering_100mi/cluster_summary.csv`
- `results/npj_100mi/figureB_nchs_milton.{pdf,png}` (appendix rural-urban code map)
- `results/npj_100mi/figureC_clusters_milton.{pdf,png}` (appendix cluster-illustration map)

In [1]:
import os, warnings
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from libpysal.weights import Queen
from scipy.sparse import lil_matrix
from scipy.sparse.csgraph import connected_components
from shapely.geometry import LineString
warnings.filterwarnings('ignore')

ROOT='/Users/qing/Library/CloudStorage/OneDrive-ColumbiaUniversityIrvingMedicalCenter/4_hurricane_category'; HO='/Users/qing/Library/CloudStorage/OneDrive-ColumbiaUniversityIrvingMedicalCenter/hurricane_oct'
COUNTY_LIST=f'{ROOT}/results/local_level/milton_100mi/counties_geoid_cut_100.txt'
ACS=f'{ROOT}/notebook/acs_socioeconomic_v2.csv'
NCHS_CSV=f'{ROOT}/data/NCHS Urban-Rural Classification Scheme for Counties.csv'
COUNTY_SHP=f'{HO}/data/county_geo/tl_2023_us_county/tl_2023_us_county.shp'
TRACK_SHP=f'{HO}/data/storm_track/milton_storm_track.shp'
OUT_CLUSTER=f'{ROOT}/results/local_level/milton_clustered_100mi'
OUT_CLUSTERING=f'{ROOT}/results/milton_clustering_100mi'
OUT_NPJ=f'{ROOT}/results/npj_100mi'
for d in (OUT_CLUSTER, f'{OUT_CLUSTERING}/figures', OUT_NPJ): os.makedirs(d, exist_ok=True)

NCHS_LABELS={1:'Large central metro',2:'Large fringe metro',3:'Medium metro',4:'Small metro',5:'Micropolitan',6:'Noncore (rural)'}
MERGE_CODES={4,5,6}   # rule B1: merge small only; NCHS 1-3 stay standalone
print('setup ok')

setup ok


In [2]:
# Load counties + ACS + NCHS + geometry + track; compute distance-to-track; Queen contiguity
geoids=[int(x.strip()) for x in open(COUNTY_LIST) if x.strip()]
acs=pd.read_csv(ACS); acs['GEOID']=acs['GEOID'].astype(int)
nchs=pd.read_csv(NCHS_CSV, encoding='utf-8-sig')
nchs['GEOID']=nchs['Location'].astype(int)
nchs['nchs_code']=nchs['2023 Code'].str.extract(r'(\d)').astype(int)

gdf=gpd.read_file(COUNTY_SHP); gdf['GEOID']=gdf['GEOID'].astype(int)
gdf=gdf[gdf['GEOID'].isin(geoids)].copy()
gdf=gdf.merge(acs[['GEOID','total_population','median_household_income','pct_no_vehicle','insurance_coverage_pct']],on='GEOID',how='left')
gdf=gdf.merge(nchs[['GEOID','nchs_code']],on='GEOID',how='left').to_crs(epsg=5070).reset_index(drop=True)
gdf['nchs_label']=gdf['nchs_code'].map(NCHS_LABELS)

trk=gpd.read_file(TRACK_SHP).to_crs(epsg=5070)
track_line=LineString(trk.geometry.tolist()) if (trk.geom_type=='Point').all() else trk.unary_union
gdf['centroid']=gdf.geometry.centroid
gdf['dist_to_track_mi']=gdf['centroid'].apply(lambda p: track_line.distance(p))/1000.0/1.60934

w=Queen.from_dataframe(gdf)
print(f'{len(gdf)} counties | Queen mean nbrs={w.mean_neighbors:.1f} | missing NCHS={gdf.nchs_code.isna().sum()}')
print(gdf.nchs_label.value_counts().sort_index().to_string())

34 counties | Queen mean nbrs=5.0 | missing NCHS=0
nchs_label
Large central metro     3
Large fringe metro      7
Medium metro           13
Micropolitan            5
Noncore (rural)         1
Small metro             5


In [3]:
# Rule B1: NCHS 1-3 standalone; NCHS 4-6 -> connected components within each NCHS code
gdf['cluster']=-1; cid=0; meta=[]
for code in sorted(gdf.nchs_code.unique()):
    idx=gdf.index[gdf.nchs_code==code]
    if code not in MERGE_CODES:                       # keep each metro county standalone
        for i in idx:
            gdf.loc[i,'cluster']=cid
            meta.append({'cluster':cid,'nchs_code':code,'nchs_label':NCHS_LABELS[code],'n_counties':1})
            cid+=1
        continue
    nmap={o:k for k,o in enumerate(idx)}
    sub=lil_matrix((len(idx),len(idx)),dtype=int)
    for i in idx:
        for j in w.neighbors[i]:
            if j in nmap: sub[nmap[i],nmap[j]]=1; sub[nmap[j],nmap[i]]=1
    ncomp,lab=connected_components(sub.tocsr(),directed=False)
    for comp in range(ncomp):
        members=idx[lab==comp]; gdf.loc[members,'cluster']=cid
        meta.append({'cluster':cid,'nchs_code':code,'nchs_label':NCHS_LABELS[code],'n_counties':int((lab==comp).sum())})
        cid+=1
n_clusters=cid
assert (gdf['cluster']>=0).all(), 'unassigned county!'
assert (gdf.groupby('cluster')['nchs_code'].nunique()==1).all(), 'mixed-NCHS cluster!'
print(f'Milton: {len(gdf)} counties -> {n_clusters} clusters (rule B1); NCHS-homogeneous OK')
print('merged clusters (>1 county):')
for m in meta:
    if m['n_counties']>1:
        names=', '.join(gdf.loc[gdf.cluster==m['cluster'],'NAME'])
        print(f"  C{m['cluster']} NCHS{m['nchs_code']} ({m['n_counties']}): {names}")

Milton: 34 counties -> 30 clusters (rule B1); NCHS-homogeneous OK
merged clusters (>1 county):
  C23 NCHS4 (2): Highlands, Charlotte
  C24 NCHS4 (2): Citrus, Sumter
  C26 NCHS5 (3): Hendry, Okeechobee, Glades


In [4]:
# Cluster summary + export assignments
summ=gdf.groupby('cluster').agg(
    n_counties=('GEOID','count'), total_pop=('total_population','sum'),
    median_income=('median_household_income','median'),
    mean_dist_to_track=('dist_to_track_mi','mean'),
    nchs_code=('nchs_code','first')).round(1)
summ['nchs_label']=summ['nchs_code'].map(NCHS_LABELS)
summ.to_csv(f'{OUT_CLUSTERING}/cluster_summary.csv')

export=gdf[['GEOID','NAME','cluster','nchs_code','nchs_label','median_household_income','total_population','dist_to_track_mi']].copy()
export=export.sort_values(['cluster','GEOID']).reset_index(drop=True)
export.to_csv(f'{OUT_CLUSTER}/county_cluster_assignments.csv', index=False)
export.to_csv(f'{OUT_CLUSTERING}/county_cluster_assignments.csv', index=False)
with open(f'{OUT_CLUSTER}/counties_geoid_cut_100.txt','w') as f:
    f.write('\n'.join(str(g) for g in export['GEOID']))
print('saved cluster assignments ->', OUT_CLUSTER)
print(summ[['n_counties','total_pop','nchs_label','median_income','mean_dist_to_track']].to_string())

saved cluster assignments -> /Users/qing/Library/CloudStorage/OneDrive-ColumbiaUniversityIrvingMedicalCenter/4_hurricane_category/results/local_level/milton_clustered_100mi
         n_counties  total_pop           nchs_label  median_income  mean_dist_to_track
cluster                                                                               
0                 1    1427403  Large central metro        72629.0                19.9
1                 1     959918  Large central metro        66406.0                41.5
2                 1    1468560  Large central metro        70612.0                24.4
3                 1     196621   Large fringe metro        59202.0                66.1
4                 1    1494805   Large fringe metro        76066.0               116.6
5                 1     278722   Large fringe metro       100020.0               108.1
6                 1     386829   Large fringe metro        66239.0                46.0
7                 1     569211   Large fring

In [5]:
# Maps. cmap + a reusable cluster-OUTLINE map: counties drawn with thin edges, CLUSTER boundaries thick,
# so merged groups are visibly outlined.
import matplotlib as mpl
mpl.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial','Helvetica','DejaVu Sans'],
    'font.size':7,'pdf.fonttype':42,'ps.fonttype':42})
cmap=plt.cm.get_cmap('RdYlGn_r',6)
trk_gdf=gpd.GeoDataFrame(geometry=[track_line],crs='EPSG:5070')

def cluster_outline_map(counties, track, title, outstem, label=True):
    num=counties.select_dtypes(include=[np.number]).columns.tolist()
    diss=counties[[c for c in num if c!='cluster']+['cluster','geometry']].dissolve(by='cluster',aggfunc='median')
    fig,ax=plt.subplots(figsize=(4.4,4.8))
    counties.plot(column='nchs_code',cmap=cmap,vmin=1,vmax=6,edgecolor='white',linewidth=0.2,ax=ax,
                  legend=True,legend_kwds={'label':'NCHS code (1=metro, 6=rural)','shrink':0.55})
    diss.boundary.plot(ax=ax,color='black',linewidth=0.9)     # cluster outlines = the merges
    track.plot(ax=ax,color='red',linewidth=1.0)
    if label:
        for cidx,row in diss.iterrows():
            c=row.geometry.centroid
            ax.annotate(f'C{cidx}',(c.x,c.y),fontsize=4.5,ha='center',va='center',
                        bbox=dict(boxstyle='round,pad=0.08',fc='white',alpha=0.65,lw=0))
    b=counties.total_bounds; p=0.06*max(b[2]-b[0],b[3]-b[1])
    ax.set_xlim(b[0]-p,b[2]+p); ax.set_ylim(b[1]-p,b[3]+p)
    ax.set_title(title,fontsize=8,loc='left'); ax.set_axis_off()
    fig.savefig(f'{OUT_NPJ}/{outstem}.pdf',bbox_inches='tight')
    fig.savefig(f'{OUT_NPJ}/{outstem}.png',dpi=300,bbox_inches='tight')
    plt.close(fig); print('saved', outstem)

# (A) NCHS per-county map (appendix figure B) — no cluster outline
figA,axA=plt.subplots(figsize=(4.2,4.6))
gdf.plot(column='nchs_code',cmap=cmap,vmin=1,vmax=6,edgecolor='white',linewidth=0.3,ax=axA,
         legend=True,legend_kwds={'label':'NCHS code (1=large metro, 6=rural)','shrink':0.6})
trk_gdf.plot(ax=axA,color='black',linewidth=1.2)
_b=gdf.total_bounds; _p=0.06*max(_b[2]-_b[0],_b[3]-_b[1])
axA.set_xlim(_b[0]-_p,_b[2]+_p); axA.set_ylim(_b[1]-_p,_b[3]+_p)
axA.set_title('Milton — rural-urban (NCHS) code by county (100 mi)',fontsize=8,loc='left'); axA.set_axis_off()
figA.savefig(f'{OUT_NPJ}/figureB_nchs_milton.pdf',bbox_inches='tight')
figA.savefig(f'{OUT_NPJ}/figureB_nchs_milton.png',dpi=300,bbox_inches='tight'); plt.close(figA)

# (C) Milton cluster-outline map (the merge illustration)
cluster_outline_map(gdf, trk_gdf, f'Milton — {n_clusters} clusters (counties outlined by cluster; rule B1, 100 mi)',
                    'figureC_clusters_milton', label=True)
print('saved figureB_nchs_milton + figureC_clusters_milton')

saved figureC_clusters_milton
saved figureB_nchs_milton + figureC_clusters_milton
